# 🏠 House Price Predictor

End-to-end regression workflow using the California Housing dataset.

We will explore the data, preprocess it, compare Linear Regression and Random Forest, evaluate the models with MAE/RMSE/R², and inspect feature importance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load the dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.head()

In [ ]:
print('Shape:', df.shape)
display(df.describe().T)
print('Missing values:')
display(df.isna().sum())

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df['MedHouseVal'], bins=40, kde=True)
plt.title('Distribution of Median House Value')
plt.xlabel('Median house value ($100,000 units)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Prepare the data

In [ ]:
features = housing.feature_names
X = df[features]
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 4. Train two regression models

In [ ]:
linear_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

forest_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1))
])

models = {'Linear Regression': linear_model, 'Random Forest': forest_model}

## 5. Evaluate the models

In [ ]:
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'R2': r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df.style.format({'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'R2': '{:.4f}'}))

In [ ]:
for name, pred in predictions.items():
    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, pred, alpha=0.35)
    lower, upper = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    plt.plot([lower, upper], [lower, upper], '--')
    plt.title(f'Actual vs Predicted — {name}')
    plt.xlabel('Actual value ($100,000 units)')
    plt.ylabel('Predicted value ($100,000 units)')
    plt.show()

## 6. Random Forest feature importance

In [ ]:
rf = forest_model.named_steps['model']
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
display(importance.to_frame('importance'))

plt.figure(figsize=(9, 5))
sns.barplot(x=importance.values, y=importance.index)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.show()

## 7. Example prediction

The target is measured in $100,000 units. Multiply a prediction by 100,000 to express it in dollars.

In [ ]:
sample = X_test.iloc[[0]]
prediction = forest_model.predict(sample)[0]
print(f'Predicted value: ${prediction * 100_000:,.0f}')
print(f'Actual value:    ${y_test.iloc[0] * 100_000:,.0f}')